# Batch ingestion: Kraken historical OHLC

This notebook extracts the latest completed hourly OHLC candles for BTCUSDT, ETHUSDT, and BNBUSDT from the public Binance API, stores a raw CSV file in a Unity Catalog Volume, and loads the data idempotently into a Bronze Delta table.

### Setup & Connecting to Kraken API

In [0]:
# Parameters for different environments

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "team_crypto_bronze")
dbutils.widgets.text("volume", "raw_data")

CATALOG = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
VOLUME = dbutils.widgets.get("volume")

RAW_DIRECTORY = (
    f"/Volumes/{CATALOG}/"
    f"{BRONZE_SCHEMA}/{VOLUME}/ohlc_history"
)

TARGET_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.ohlc_batch"
)

API_URL = "https://api.kraken.com/0/public/OHLC"

SYMBOLS = [
    "XBTUSD",
    "ETHUSD",
    "SOLUSD"
]

INTERVAL = 60

print("Raw directory:", RAW_DIRECTORY)
print("Target table:", TARGET_TABLE)

In [0]:
import requests

test_response = requests.get(
    API_URL,
    params={
        "pair": "XBTUSD",
        "interval": INTERVAL
    },
    timeout=30
)

print("Status code:", test_response.status_code)
print("Request URL:", test_response.url)

test_response.raise_for_status()
test_data = test_response.json()

print("API errors:", test_data["error"])
print("Result keys:", test_data["result"].keys())

In [0]:
import requests

SYMBOL_MAPPING = {
    "XBTUSD": "BTCUSD",
    "ETHUSD": "ETHUSD",
    "SOLUSD": "SOLUSD"
}

records = []

for kraken_symbol, canonical_symbol in SYMBOL_MAPPING.items():
    response = requests.get(
        API_URL,
        params={
            "pair": kraken_symbol,
            "interval": INTERVAL
        },
        timeout=30
    )

    response.raise_for_status()
    payload = response.json()

    if payload["error"]:
        raise RuntimeError(
            f"Kraken API error for {kraken_symbol}: {payload['error']}"
        )

    # remove the last candle; it may still be forming and changing
    result_key = next(
        key for key in payload["result"]
        if key != "last"
    )

    candles = payload["result"][result_key]

    
    for candle in candles[:-1]:
        records.append({
            "symbol": canonical_symbol,
            "timestamp_unix": int(candle[0]),
            "open": candle[1], #These are prices, and they can have many decimal places. 
            "high": candle[2], #Converting to float may result in a loss of precision.
            "low": candle[3],
            "close": candle[4],
            "vwap": candle[5],
            "volume": candle[6],
            "trade_count": int(candle[7])
        })

    print(f"{canonical_symbol}: {len(candles) - 1} completed candles")

print("Total records:", len(records))

In [0]:
ohlc_df = spark.createDataFrame(records)

display(ohlc_df.limit(10))

### Creating raw data as CSV File in Unity Catalog Volume

In [0]:
import csv
RAW_FILE_PATH = f"{RAW_DIRECTORY}/ohlc_history.csv"
dbutils.fs.mkdirs(RAW_DIRECTORY)

COLUMN_NAMES = [
    "symbol",
    "timestamp_unix",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "volume",
    "trade_count"
]

with open(
    RAW_FILE_PATH,
    mode="w",
    newline="",
    encoding="utf-8"
) as csv_file:

    writer = csv.DictWriter(
        csv_file,
        fieldnames=COLUMN_NAMES
    )

    writer.writeheader()
    writer.writerows(records)

print("Path:", RAW_FILE_PATH)
print("Saved rows:", len(records))

In [0]:
raw_check_df = (
    spark.read
    .option("header", True)
    .csv(RAW_FILE_PATH)
)

print("Rows in CSV:", raw_check_df.count())

display(raw_check_df.limit(10))

### Converted columns to the correct types and added metadata

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType,
    StringType
    )

#so that the time is processed in the same way
spark.conf.set("spark.sql.session.timeZone", "UTC")

ohlc_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("timestamp_unix", LongType(), False),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("vwap", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("trade_count", LongType(), True)
])

bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(ohlc_schema)
    .load(RAW_FILE_PATH)

      .select(
        "*",
        F.col("_metadata.file_name").alias("source_filename")
    )
      .withColumn(
        "event_timestamp",
        F.to_timestamp(F.from_unixtime("timestamp_unix"))
    )

    
    .withColumn("source", F.lit("kraken_api"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

display(bronze_df.limit(10))

In [0]:
bronze_df.printSchema()
print("Rows prepared for Bronze:", bronze_df.count())

### Check for duplicates before bronze layer and idempotent write

In [0]:

source_rows = bronze_df.count()
unique_rows = (
    bronze_df
    .select("symbol", "timestamp_unix")
    .distinct()
    .count()
)

duplicate_rows = source_rows - unique_rows

print("Source rows:", source_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", duplicate_rows)

In [0]:
bronze_dedup_df = bronze_df.dropDuplicates(
    ["symbol", "timestamp_unix"]
)

### Idempotent write in Bronze

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(TARGET_TABLE):
    before_count = spark.table(TARGET_TABLE).count()
    target_delta = DeltaTable.forName(
        spark,
        TARGET_TABLE
    )

    (
        target_delta.alias("target")
        .merge(
            bronze_dedup_df.alias("source"),
            """
            target.symbol = source.symbol
            AND target.timestamp_unix = source.timestamp_unix
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    action = "MERGE completed"

else:

    before_count = 0

    (
        bronze_dedup_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )

    action = "Bronze table created"
# before it was created in 1st cell: 
# TARGET_TABLE = (f"{CATALOG}.{BRONZE_SCHEMA}.ohlc_batch")

after_count = spark.table(TARGET_TABLE).count()

print(action)
print("Target table:", TARGET_TABLE)
print("Rows before:", before_count)
print("Rows after:", after_count)
print("New rows inserted:", after_count - before_count)

### Quality checks

In [0]:
target_df = spark.table(TARGET_TABLE)

#check for duplicates after bronze layer
duplicate_count = (
    target_df
    .groupBy("symbol", "timestamp_unix")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

#check for obligatory columns
invalid_count = (
    target_df
    .filter(F.col("symbol").isNull() | F.col("timestamp_unix").isNull() | F.col("close").isNull())
    .count()
)

print("Total Bronze rows:", target_df.count())
print("Duplicate keys:", duplicate_count)
print("Invalid rows:", invalid_count)

assert duplicate_count == 0, "Duplicate records detected"
assert invalid_count == 0, "Invalid records detected"

print("Data quality checks passed")

In [0]:
display(
    target_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("records"),
        F.min("event_timestamp").alias("first_timestamp"),
        F.max("event_timestamp").alias("last_timestamp"),
        F.min("close").alias("minimum_close"),
        F.max("close").alias("maximum_close")
    )
    .orderBy("symbol")
)